# KGE Triplet Extraction — Pure GLiNER (No Vocabularies)

Extract triplets `(model, dataset, metric)` from evaluation tables in KGE papers using **only GLiNER** for entity recognition. No hardcoded vocabularies, normalization maps, or keyword lists.

**Pipeline:**
1. Setup (imports, paths)
2. Extract tables from PDFs with deepdoctection (cached)
3. Load GLiNER model
4. Detect entities row-by-row with GLiNER → form triplets
5. Export to Excel

In [ ]:
# ─── Section 1 — Setup: imports and paths ─────────────────────────────────────
import json
import re
import warnings
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore")

SHOW_VERBOSE = False


def _find(candidates):
    return next((p for p in candidates if p.exists()), None)


PDF_DIR = _find([Path("pdfs_prueba"), Path("table_extraction/pdfs_prueba")])
assert PDF_DIR is not None, "Cannot locate pdfs_prueba/"

PDF_FILES = sorted(PDF_DIR.glob("*.pdf"))

print(f"PDF directory : {PDF_DIR}")
print(f"PDFs found    : {len(PDF_FILES)}")
for p in PDF_FILES:
    print(f"  • {p.name}")

## Section 4 — Table Extraction with deepdoctection

Run deepdoctection on each PDF (configuration: `USE_OCR=False`, `USE_PDF_MINER=True` — fastest, uses the text layer).  
Results are **cached** as `<stem>_dd.json` inside `pdfs_prueba/` so subsequent runs skip processing.

In [ ]:
# ─── Section 2 — deepdoctection extraction (with JSON cache) ─────────────────

def run_deepdoctection(path_pdf: Path, cache_dir: Path, verbose: bool = False) -> dict:
    """Analyze one PDF with deepdoctection and return structured results.
    Caches output as <stem>_dd.json so subsequent runs are instant."""
    cache_file = cache_dir / f"{path_pdf.stem}_dd.json"

    if cache_file.exists():
        with open(cache_file, "r", encoding="utf-8") as f:
            result = json.load(f)
        if verbose:
            n = sum(len(p["tables"]) for p in result["results"])
            print(f"[CACHE] {path_pdf.name} ({n} tables)")
        return result

    if verbose:
        print(f"[RUN] {path_pdf.name}")

    import deepdoctection as dd
    analyzer = dd.get_dd_analyzer(
        config_overwrite=["USE_OCR=False", "USE_PDF_MINER=True"]
    )
    df_dd = analyzer.analyze(path=str(path_pdf))
    df_dd.reset_state()

    results_data = []
    for dp in df_dd:
        tables = [{"csv": t.csv, "html": t.html} for t in dp.tables]
        if tables:
            results_data.append({"page": dp.page_number + 1, "tables": tables})

    result = {"file_name": str(path_pdf), "results": results_data}
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    return result

dd_outputs: dict = {}
extraction_rows = []

for pdf in PDF_FILES:
    cache_file = PDF_DIR / f"{pdf.stem}_dd.json"
    source = "cache" if cache_file.exists() else "run"
    dd_outputs[pdf.stem] = run_deepdoctection(pdf, cache_dir=PDF_DIR, verbose=SHOW_VERBOSE)
    n_tables = sum(len(p["tables"]) for p in dd_outputs[pdf.stem]["results"])
    extraction_rows.append({"paper": pdf.stem, "source": source, "tables": n_tables})

extract_df = pd.DataFrame(extraction_rows).sort_values(["tables", "paper"], ascending=[False, True])
print(f"\nExtracted tables from {len(PDF_FILES)} PDFs (total: {int(extract_df['tables'].sum())})")
display(extract_df)

## Section 3 — GLiNER Dependency Check

Load GLiNER model dependencies for triplet extraction.

In [ ]:
# ─── Section 3 — Optional GLiNER dependency check ─────────────────────────────
import sys, subprocess

AUTO_INSTALL_GLINER = False
GLINER_INSTALL_SPECS = [
    "gliner==0.2.26",
    "transformers>=4.51.3,<5.2",
    "tokenizers>=0.21,<0.22",
    "huggingface_hub>=0.34,<1.0",
]

if AUTO_INSTALL_GLINER:
    cmd = [sys.executable, "-m", "pip", "install", *GLINER_INSTALL_SPECS]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)

from gliner import GLiNER
import gliner
print(f"GLiNER available: {getattr(gliner, '__version__', 'unknown version')}")

In [ ]:
# ─── Section 4 — Pure GLiNER triplet extraction (no vocabularies) ────────────

GLINER_MODEL_ID = "urchade/gliner_medium-v2.1"
GLINER_LABELS   = ["model", "dataset", "metric"]
GLINER_MIN_SCORE = 0.35

gliner_model = GLiNER.from_pretrained(GLINER_MODEL_ID)


# ── Helpers ──────────────────────────────────────────────────────────────────

def _rows_from_html(html: str) -> list[str]:
    """One text string per <tr>, cells joined by ' | '."""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for tr in soup.find_all("tr"):
        cells = [c.get_text(separator=" ", strip=True) for c in tr.find_all(["th", "td"])]
        cells = [c for c in cells if c]
        if cells:
            rows.append(" | ".join(cells))
    return rows


def _clean_entity(value: str) -> str:
    """Light cleanup: strip citation markers and whitespace."""
    s = str(value).strip()
    s = re.sub(r"\[[^\]]{1,50}\]", "", s)                       # [1], [Smith 2020]
    s = re.sub(r"\((?:[^\)]*\d{4}[^\)]*|[^\)]*et\s*al\.?[^\)]*)\)", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s+", " ", s).strip(" .,:;-")
    return s


def _extract_entities(text: str) -> dict[str, list[str]]:
    """Run GLiNER on a text fragment and return detected entities."""
    out = {"model": [], "dataset": [], "metric": []}
    if not text.strip():
        return out
    try:
        entities = gliner_model.predict_entities(text[:3000], GLINER_LABELS)
    except Exception:
        return out

    found: dict[str, set[str]] = {"model": set(), "dataset": set(), "metric": set()}
    for ent in entities:
        label = str(ent.get("label", "")).strip().lower()
        score = float(ent.get("score", 0.0) or 0.0)
        value = _clean_entity(str(ent.get("text", "")))
        if score < GLINER_MIN_SCORE or not value or len(value) < 2:
            continue
        # Skip pure numbers (metric *values*, not names)
        if re.fullmatch(r"[+-]?\d+(?:\.\d+)?", value):
            continue
        if label in found:
            found[label].add(value)

    return {k: sorted(v) for k, v in found.items()}


def _triplets_from_entities(ents: dict[str, list[str]]) -> list[tuple[str, str, str]]:
    return [(m, d, mt) for m in ents["model"] for d in ents["dataset"] for mt in ents["metric"]]


# ── Main extraction loop ────────────────────────────────────────────────────

triplet_rows: list[dict] = []
table_meta_rows: list[dict] = []

for pdf_stem, dd_result in dd_outputs.items():
    for page_data in dd_result.get("results", []):
        page_num = int(page_data.get("page", 0) or 0)
        for table_idx, table in enumerate(page_data.get("tables", []), start=1):
            table_name = f"{pdf_stem}_p{page_num}_t{table_idx}"
            html = table.get("html", "")
            if not html:
                continue

            row_texts = _rows_from_html(html)
            if not row_texts:
                continue

            table_triplets: set[tuple[str, str, str]] = set()
            all_models, all_datasets, all_metrics = set(), set(), set()

            # --- Row-by-row pass ---
            for row_text in row_texts:
                ents = _extract_entities(row_text)
                all_models.update(ents["model"])
                all_datasets.update(ents["dataset"])
                all_metrics.update(ents["metric"])
                for t in _triplets_from_entities(ents):
                    table_triplets.add(t)

            # --- Fallback: full table text ---
            if not table_triplets:
                full_text = "\n".join(row_texts)
                ents_full = _extract_entities(full_text)
                all_models.update(ents_full["model"])
                all_datasets.update(ents_full["dataset"])
                all_metrics.update(ents_full["metric"])
                for t in _triplets_from_entities(ents_full):
                    table_triplets.add(t)

            # --- Context propagation: combine table-level entities with row models ---
            # If rows detected models but missed datasets/metrics that the table overall has,
            # generate additional combinations.
            if all_models and (all_datasets or all_metrics):
                for t in _triplets_from_entities({
                    "model": sorted(all_models),
                    "dataset": sorted(all_datasets),
                    "metric": sorted(all_metrics),
                }):
                    table_triplets.add(t)

            table_meta_rows.append({
                "paper": pdf_stem,
                "table_name": table_name,
                "models": " | ".join(sorted(all_models)) or "(none)",
                "datasets": " | ".join(sorted(all_datasets)) or "(none)",
                "metrics": " | ".join(sorted(all_metrics)) or "(none)",
                "triplets_found": len(table_triplets),
            })

            for m, d, mt in sorted(table_triplets):
                triplet_rows.append({
                    "paper": pdf_stem,
                    "table_name": table_name,
                    "model": m,
                    "dataset": d,
                    "metric": mt,
                })

# ── Build dataframes ────────────────────────────────────────────────────────

triplets_df = pd.DataFrame(triplet_rows) if triplet_rows else pd.DataFrame(
    columns=["paper", "table_name", "model", "dataset", "metric"]
)
triplets_df = triplets_df.drop_duplicates().sort_values(
    ["paper", "table_name", "model", "dataset", "metric"]
).reset_index(drop=True)

table_meta_df = pd.DataFrame(table_meta_rows)

print("=" * 70)
print("PURE GLINER TRIPLET EXTRACTION")
print("=" * 70)
print(f"Total triplets : {len(triplets_df)}")
print(f"Unique models  : {triplets_df['model'].nunique() if len(triplets_df) else 0}")
print(f"Unique datasets: {triplets_df['dataset'].nunique() if len(triplets_df) else 0}")
print(f"Unique metrics : {triplets_df['metric'].nunique() if len(triplets_df) else 0}")
print()
display(triplets_df)
print()
print("Table metadata:")
display(table_meta_df)

In [ ]:
# ─── Tables with metrics: full detail (triplets if available, partial otherwise)

metric_table_rows = []

# Tables that have at least one metric detected
tables_with_metrics = table_meta_df[table_meta_df["metrics"] != "(none)"].copy()

for _, meta in tables_with_metrics.iterrows():
    paper = meta["paper"]
    tname = meta["table_name"]

    # Get triplets for this table (if any)
    table_trips = triplets_df[
        (triplets_df["paper"] == paper) & (triplets_df["table_name"] == tname)
    ]

    if not table_trips.empty:
        # Table has full triplets -> one row per triplet
        for _, trip in table_trips.iterrows():
            metric_table_rows.append({
                "paper": paper,
                "table_name": tname,
                "model": trip["model"],
                "dataset": trip["dataset"],
                "metric": trip["metric"],
                "has_triplet": True,
            })
    else:
        # Table has metrics but no complete triplet -> include partial info
        models = meta["models"] if meta["models"] != "(none)" else ""
        datasets = meta["datasets"] if meta["datasets"] != "(none)" else ""
        metrics_list = [m.strip() for m in meta["metrics"].split("|")]
        model_list = [m.strip() for m in models.split("|")] if models else [""]
        dataset_list = [d.strip() for d in datasets.split("|")] if datasets else [""]

        for metric in metrics_list:
            for model in model_list:
                for dataset in dataset_list:
                    metric_table_rows.append({
                        "paper": paper,
                        "table_name": tname,
                        "model": model,
                        "dataset": dataset,
                        "metric": metric,
                        "has_triplet": False,
                    })

tables_with_metrics_df = pd.DataFrame(metric_table_rows)
if not tables_with_metrics_df.empty:
    tables_with_metrics_df = tables_with_metrics_df.drop_duplicates().sort_values(
        ["paper", "table_name", "model", "dataset", "metric"]
    ).reset_index(drop=True)

n_with = int(tables_with_metrics_df["has_triplet"].sum()) if len(tables_with_metrics_df) else 0
n_without = len(tables_with_metrics_df) - n_with

print(f"Tables with metrics: {len(tables_with_metrics)}")
print(f"  Rows with full triplet    : {n_with}")
print(f"  Rows partial (no triplet) : {n_without}")
print()
display(tables_with_metrics_df)

In [ ]:
# ─── Section 5 — Export to Excel ─────────────────────────────────────────────
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

export_file = PDF_DIR / "gliner_triplets.xlsx"

with pd.ExcelWriter(export_file, engine="openpyxl") as writer:
    triplets_df.to_excel(writer, index=False, sheet_name="Triplets")
    table_meta_df.to_excel(writer, index=False, sheet_name="Table Metadata")
    tables_with_metrics_df.to_excel(writer, index=False, sheet_name="Tables With Metrics")

    wb = writer.book
    header_fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")
    header_font = Font(bold=True)

    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.font = header_font
            cell.fill = header_fill
        for col_idx, col_cells in enumerate(
            ws.iter_cols(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column), start=1
        ):
            max_len = max(len(str(c.value)) if c.value is not None else 0 for c in col_cells)
            ws.column_dimensions[get_column_letter(col_idx)].width = min(max(10, max_len + 2), 60)

print(f"Excel: {export_file}")
print(f"  Triplets sheet          : {len(triplets_df)} rows")
print(f"  Metadata sheet          : {len(table_meta_df)} rows")
print(f"  Tables With Metrics sheet: {len(tables_with_metrics_df)} rows")